In [2]:
# ============================================================
# 06_R_cpu_anvl_post_estimation.ipynb
# ANVL CPU post-estimation benchmark for the Apollo Swiss
# route-choice MXL model
#
# This notebook benchmarks CPU-based estimation and
# post-estimation for the panel mixed-logit specification used
# in the Apollo Swiss route-choice example. ANVL provides the
# XLA-backed likelihood and analytical score evaluator, while BGW is
# used for maximum-likelihood estimation.
#
# Scope:
# - CPU execution through ANVL with the PJRT CPU backend.
# - Deterministic Halton draws for four lognormal coefficients.
# - BGW estimation using individual likelihoods and analytical scores.
# - Post-estimation through postestimation_bgw().
# - Draw sizes of 500, 1,000, 2,000, 5,000, and 10,000.
# - Separate timing of JIT compilation, likelihood evaluation,
#   estimation, and post-estimation.
# - Post-estimation coefficient tables reported by draw size.
# - CPU, package, and R session records for reproducibility.
#
# Main outputs:
# - results/benchmark_summary_anvl_bgw_cpu.xlsx
# - results/session_info_anvl_cpu.txt
# - Printed post-estimation coefficient tables for each draw size.
# ============================================================


In [3]:
# -------------------------------------------------------------------
# 1. Packages
# -------------------------------------------------------------------
# Packages are installed only when missing. The benchmark depends on
# BGW for optimisation, and randtoolbox for Halton draws.

apt_pkgs <- c(
  "build-essential", "cmake", "pkg-config",
  "libprotobuf-dev", "libprotoc-dev", "protobuf-compiler",
  "libuv1-dev", "libcurl4-openssl-dev", "libssl-dev", "libxml2-dev"
)

stopifnot(system2("apt-get", c("update", "-qq")) == 0L)
stopifnot(system2("apt-get", c("install", "-y", apt_pkgs)) == 0L)

# 2. R packages

Sys.setenv(PJRT_INSTALL = "1")

options(repos = c(
  "https://r-xla.r-universe.dev",
  "https://mlverse.r-universe.dev",
  "https://cloud.r-project.org"
))

pkgs <- c(
  "randtoolbox", "bgw", "maxLik",
  "pjrt", "anvl"
)

for (pkg in pkgs) {
  if (!requireNamespace(pkg, quietly = TRUE)) {
    install.packages(pkg)
  }
}

# 3. Load packages

suppressPackageStartupMessages({
  library(randtoolbox)
  library(bgw)
  library(maxLik)
  library(pjrt)
  library(anvl)
})


Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

also installing the dependency ‘rngWELL’


Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

also installing the dependencies ‘zoo’, ‘miscTools’, ‘sandwich’


Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

also installing the dependencies ‘checkmate’, ‘safetensors’, ‘tengen’


Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

also installing the dependencies ‘stablehlo’, ‘xlamisc’




In [4]:
# ============================================================
# Fast Apollo installation in Colab using Posit Package Manager
# ============================================================

options(HTTPUserAgent = sprintf(
  "R/%s R (%s)",
  getRversion(),
  paste(
    getRversion(),
    R.version["platform"],
    R.version["arch"],
    R.version["os"]
  )
))

# Auto-detect Ubuntu codename, e.g., jammy or noble
ubuntu_codename <- tryCatch(
  system("bash -lc '. /etc/os-release; echo ${VERSION_CODENAME}'", intern = TRUE),
  error = function(e) "jammy"
)

ppm_repo <- sprintf(
  "https://packagemanager.posit.co/cran/__linux__/%s/latest",
  ubuntu_codename[1]
)

options(repos = c(CRAN = ppm_repo))

cat("Using CRAN repo:\n", getOption("repos"), "\n\n")

install.packages("apollo", Ncpus = parallel::detectCores())

library(apollo)

pkg_versions <- data.frame(
  package = c("apollo", "randtoolbox", "bgw", "maxLik", "pjrt", "anvl"),
  version = vapply(
    c("apollo", "randtoolbox", "bgw", "maxLik", "pjrt", "anvl"),
    function(pkg) as.character(packageVersion(pkg)),
    character(1)
  )
)

print(pkg_versions)

Using CRAN repo:
 https://packagemanager.posit.co/cran/__linux__/jammy/latest 



Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

also installing the dependencies ‘listenv’, ‘parallelly’, ‘future’, ‘globals’, ‘SparseM’, ‘MatrixModels’, ‘truncnorm’, ‘future.apply’, ‘mcmc’, ‘quantreg’, ‘mnormt’, ‘mvtnorm’, ‘numDeriv’, ‘Deriv’, ‘matrixStats’, ‘coda’, ‘Rsolnp’, ‘MCMCpack’, ‘RcppArmadillo’, ‘RcppEigen’




             . ,,                                                            
            ,      ,,                                                        
 ,,,,,,    ,         ,,                                                      
,     ,,  ,            ,,,,.                                                 
,,     , ,,   ,,,,,,    ,,,                                 //  //           
  ,     ,,,.   ,,,,,.   ,,      ////                        //  //           
,,     ,,,,,.           ,,     // //     //////    /////    //  //    /////  
,,,        ,,           ,      //  //    /    //  //   //   //  //   //   // 
              ,,  

                package version
apollo           apollo   0.3.9
randtoolbox randtoolbox   2.0.5
bgw                 bgw   0.1.4
maxLik           maxLik 1.5.2.2
pjrt               pjrt   0.5.0
anvl               anvl   0.4.0


In [5]:
# -------------------------------------------------------------------
# 2. Project paths
# -------------------------------------------------------------------

proj_root <- getwd()
raw_results_dir <- file.path(proj_root, "results", "raw")
summary_file <- file.path(proj_root, "results", "benchmark_summary_anvl_bgw_cpu.xlsx")
session_info_file <- file.path(proj_root, "results", "session_info_anvl_cpu.txt")

dir.create(raw_results_dir, recursive = TRUE, showWarnings = FALSE)

benchmark_results <- list()


In [6]:
# ============================================================
# 3. Define benchmark settings
# ============================================================

# This section defines common settings and input builders for
# ANVL + maxLik + CPU and ANVL + BGW + CPU.

# ------------------------------------------------------------
# 3.1 Benchmark settings
# ------------------------------------------------------------

seed <- 123L
# Standard deterministic Halton draws; seed is not used.
device_use <- "cpu"

# ------------------------------------------------------------
# 3.2 Load data and build ANVL database
# ------------------------------------------------------------
# Create ANVL data arrays with dimension task × individual × draw.

database <- apollo_swissRouteChoiceData
database <- database[order(database$ID), ]
row.names(database) <- NULL

n_indiv <- length(unique(database$ID))
n_task <- as.integer(nrow(database) / n_indiv)
n_obs <- nrow(database)

make_database_nv <- function() {

  make_x <- function(x) {
    nv_array(
      array(
        as.numeric(x),
        dim = c(n_task, n_indiv, 1L)
      ),
      dtype = "f64",
      device = device_use
    )
  }

  list(
    tt1 = make_x(database$tt1),
    tc1 = make_x(database$tc1),
    hw1 = make_x(database$hw1),
    ch1 = make_x(database$ch1),

    tt2 = make_x(database$tt2),
    tc2 = make_x(database$tc2),
    hw2 = make_x(database$hw2),
    ch2 = make_x(database$ch2),

    choice1 = make_x(database$choice == 1),
    choice2 = make_x(database$choice == 2)
  )
}

# ------------------------------------------------------------
# 3.3 Generate Halton normal draws
# ------------------------------------------------------------
# Create ANVL draw arrays with dimension task × individual × draw.

make_draws_nv <- function(num_draws) {

  halton_sequence <- halton(
    n   = n_indiv * num_draws,
    dim = 4
  )

  norm_value <- qnorm(halton_sequence)

  make_draw <- function(z) {

    draw_ind <- matrix(
      z,
      nrow = n_indiv,
      ncol = num_draws,
      byrow = TRUE
    )

    nv_array(
      array(
        as.numeric(draw_ind),
        dim = c(1L, n_indiv, num_draws)
      ),
      dtype = "f64",
      device = device_use
    )
  }

  list(
    tt = make_draw(norm_value[, 1]),
    tc = make_draw(norm_value[, 2]),
    hw = make_draw(norm_value[, 3]),
    ch = make_draw(norm_value[, 4])
  )
}

# ------------------------------------------------------------
# 3.4 Print benchmark information
# ------------------------------------------------------------

cat("Benchmark settings\n")
cat("Seed         :", seed, "\n")
cat("Device       :", device_use, "\n")
cat("Individuals  :", n_indiv, "\n")
cat("Tasks/person :", n_task, "\n")
cat("Observations :", n_obs, "\n")
cat("Raw output   :", raw_results_dir, "\n")
cat("Summary file :", summary_file, "\n")

# ------------------------------------------------------------
# 3.5 Verify CPU backend and save environment information
# ------------------------------------------------------------

cat("\nANVL device check:\n")
print(nv_array(1, dtype = "f64", device = device_use))

cpu_info <- tryCatch(
  system("lscpu", intern = TRUE),
  error = function(e) paste("lscpu failed:", conditionMessage(e))
)

cat("\nCPU information:\n")
cat(paste(cpu_info, collapse = "\n"), "\n")

sink(session_info_file)
cat("R session information\n")
print(sessionInfo())

cat("\nPackage versions\n")
for (pkg in c("apollo", "randtoolbox", "bgw", "maxLik", "pjrt", "anvl")) {
  cat(pkg, ": ", as.character(packageVersion(pkg)), "\n", sep = "")
}

cat("\nANVL device check\n")
print(nv_array(1, dtype = "f64", device = device_use))

cat("\nCPU information\n")
cat(paste(cpu_info, collapse = "\n"), "\n")
sink()

cat("Session and hardware information saved to:", session_info_file, "\n")


Benchmark settings
Seed         : 123 
Device       : cpu 
Individuals  : 388 
Tasks/person : 9 
Observations : 3492 
Raw output   : /content/results/raw 
Summary file : /content/results/benchmark_summary_anvl_bgw_cpu.xlsx 

ANVL device check:


<https://github.com/zml/pjrt-artifacts/releases/download/v14.0.1/pjrt-cpu_linux-amd64.tar.gz>


AnvlArray
 1
[ CPUf64{1} ] 

CPU information:
Architecture:                            x86_64
CPU op-mode(s):                          32-bit, 64-bit
Address sizes:                           46 bits physical, 48 bits virtual
Byte Order:                              Little Endian
CPU(s):                                  8
On-line CPU(s) list:                     0-7
Vendor ID:                               GenuineIntel
Model name:                              Intel(R) Xeon(R) CPU @ 2.20GHz
CPU family:                              6
Model:                                   79
Thread(s) per core:                      2
Core(s) per socket:                      4
Socket(s):                               1
Stepping:                                0
BogoMIPS:                                4399.99
Flags:                                   fpu vme de pse tsc msr pae mce cx8 apic sep mtrr pge mca cmov pat pse36 clflush mmx fxsr sse sse2 ss ht syscall nx pdpe1gb rdtscp lm constant_tsc rep_good no

In [ ]:
# ============================================================
# 4. Define ANVL + BGW likelihood function
# ============================================================

# This part of the code has been omitted temporarily and will be released upon publication of the corresponding paper.

In [8]:
# ============================================================
# 5. BGW estimation
# ============================================================
# Run BGW with individual likelihoods and analytical scores.

estimate_bgw <- function(calcR, calcJ, start_value) {
  bgw::bgw_mle(
    calcR = calcR,
    calcJ = calcJ,
    betaStart = start_value,
    bgw_settings = list(vcHessianMethod = "bhhh")
  )
}

In [ ]:
# ============================================================
# 6. BGW post-estimation
# ============================================================
# Recompute the Hessian from the summed analytical score and
# form conventional and sandwich-robust covariance matrices.

# This part of the code has been omitted temporarily and will be released upon publication of the corresponding paper.

In [10]:
# ============================================================
# 7.1 500-draw
# ============================================================

num_draws <- 500L

bgw_functions <- make_calcRJ_bgw(num_draws)
calcR_bgw <- bgw_functions$calcR
calcJ_bgw <- bgw_functions$calcJ

jit_time <- system.time({
  invisible(calcR_bgw(start_value_bgw))
  invisible(calcJ_bgw(start_value_bgw))
})[["elapsed"]]

likelihood_eval_time <- system.time({
  initial_prob <- calcR_bgw(start_value_bgw)
})[["elapsed"]]
initial_loglik <- sum(log(initial_prob))

estimation_time <- system.time({
  result_bgw <- estimate_bgw(
    calcR = calcR_bgw,
    calcJ = calcJ_bgw,
    start_value = start_value_bgw
  )
})[["elapsed"]]

stopifnot(isTRUE(result_bgw$hasAnalyticGrad))

postestimation_time <- system.time({
  post_bgw <- postestimation_bgw(
    result = result_bgw,
    calcJ = calcJ_bgw
  )
})[["elapsed"]]

final_loglik <- sum(log(calcR_bgw(result_bgw$estimate)))

timing_bgw <- data.frame(
  draws = num_draws,
  jit_time_sec = jit_time,
  likelihood_eval_time_sec = likelihood_eval_time,
  initial_loglik = initial_loglik,
  final_loglik = final_loglik,
  estimation_time_sec = estimation_time,
  postestimation_time_sec = postestimation_time
)

benchmark_results[[as.character(num_draws)]] <- timing_bgw

print(timing_bgw)
post_bgw$coefficients



BGW using analytic model derivatives supplied by caller...

Valid user-provided bgw_setting names(s) detected...

User-provided value for bgw_settings[["vcHessianMethod]]" is the same as default ("bhhh").

            BetaName InitialBeta(i) D(i)
    1    mu_log_b_tt          -3.00    0
    2 sigma_log_b_tt           0.01    0
    3    mu_log_b_tc          -3.00    0
    4 sigma_log_b_tc           0.01    0
    5    mu_log_b_hw          -3.00    0
    6 sigma_log_b_hw           0.01    0
    7    mu_log_b_ch          -3.00    0
    8 sigma_log_b_ch           0.01    0


    it    nf     F            RELDF    PRELDF    RELDX    MODEL stppar   D*step   NPRELDF
     0     1 2.253777958e+03
     1     2 1.775420248e+03 2.122e-01 9.248e-03 1.97e-03   G   2.26e+01 9.30e-01 5.390e-02
     2     3 1.763783129e+03 6.555e-03 3.857e-02 8.48e-02   G   0.00e+00 2.65e+01 3.857e-02
     3     4 1.568507992e+03 1.107e-01 4.733e-02 1.75e-01   G   0.00e+00 2.44e+01 4.733e-02
     4     5 1.503246130e+0

,Estimate,s.e.,t.rat.(0),Rob.s.e.,Rob.t.rat.(0)
mu_log_b_tt,-1.9961784,0.09022588,-22.124233,0.12227511,-16.325305
sigma_log_b_tt,0.4800613,0.07174709,6.691021,0.06593502,7.280824
mu_log_b_tc,-1.0298064,0.15127270,-6.807616,0.21132261,-4.873148
sigma_log_b_tc,-0.9498877,0.10571190,-8.985627,0.13088675,-7.257325
mu_log_b_hw,-2.9450102,0.08497441,-34.657612,0.09104746,-32.345882
sigma_log_b_hw,0.8180352,0.11321687,7.225383,0.12724655,6.428742
mu_log_b_ch,0.6270718,0.07579239,8.273546,0.08650689,7.248808
sigma_log_b_ch,-0.8480196,0.12900898,-6.573338,0.17671292,-4.798854


In [11]:
# ============================================================
# 7.2 1000-draw
# ============================================================

num_draws <- 1000L

bgw_functions <- make_calcRJ_bgw(num_draws)
calcR_bgw <- bgw_functions$calcR
calcJ_bgw <- bgw_functions$calcJ

jit_time <- system.time({
  invisible(calcR_bgw(start_value_bgw))
  invisible(calcJ_bgw(start_value_bgw))
})[["elapsed"]]

likelihood_eval_time <- system.time({
  initial_prob <- calcR_bgw(start_value_bgw)
})[["elapsed"]]
initial_loglik <- sum(log(initial_prob))

estimation_time <- system.time({
  result_bgw <- estimate_bgw(
    calcR = calcR_bgw,
    calcJ = calcJ_bgw,
    start_value = start_value_bgw
  )
})[["elapsed"]]

stopifnot(isTRUE(result_bgw$hasAnalyticGrad))

postestimation_time <- system.time({
  post_bgw <- postestimation_bgw(
    result = result_bgw,
    calcJ = calcJ_bgw
  )
})[["elapsed"]]

final_loglik <- sum(log(calcR_bgw(result_bgw$estimate)))

timing_bgw <- data.frame(
  draws = num_draws,
  jit_time_sec = jit_time,
  likelihood_eval_time_sec = likelihood_eval_time,
  initial_loglik = initial_loglik,
  final_loglik = final_loglik,
  estimation_time_sec = estimation_time,
  postestimation_time_sec = postestimation_time
)

benchmark_results[[as.character(num_draws)]] <- timing_bgw

print(timing_bgw)
post_bgw$coefficients



BGW using analytic model derivatives supplied by caller...

Valid user-provided bgw_setting names(s) detected...

User-provided value for bgw_settings[["vcHessianMethod]]" is the same as default ("bhhh").

            BetaName InitialBeta(i) D(i)
    1    mu_log_b_tt          -3.00    0
    2 sigma_log_b_tt           0.01    0
    3    mu_log_b_tc          -3.00    0
    4 sigma_log_b_tc           0.01    0
    5    mu_log_b_hw          -3.00    0
    6 sigma_log_b_hw           0.01    0
    7    mu_log_b_ch          -3.00    0
    8 sigma_log_b_ch           0.01    0


    it    nf     F            RELDF    PRELDF    RELDX    MODEL stppar   D*step   NPRELDF
     0     1 2.253776970e+03
     1     2 1.782933462e+03 2.089e-01 9.552e-03 1.89e-03   G   2.35e+01 9.26e-01 5.409e-02
     2     3 1.726096292e+03 3.188e-02 3.717e-02 9.09e-02   G   0.00e+00 2.73e+01 3.717e-02
     3     5 1.522125394e+03 1.182e-01 4.744e-02 1.64e-01  S-G  0.00e+00 2.57e+01 4.744e-02
     4     7 1.490489763e+0

,Estimate,s.e.,t.rat.(0),Rob.s.e.,Rob.t.rat.(0)
mu_log_b_tt,-1.9987306,0.08826220,-22.645375,0.11330808,-17.639788
sigma_log_b_tt,0.4732833,0.08251714,5.735575,0.08334634,5.678513
mu_log_b_tc,-1.0542861,0.13158661,-8.012108,0.16151510,-6.527477
sigma_log_b_tc,-1.0006019,0.08070738,-12.397898,0.07234491,-13.830992
mu_log_b_hw,-2.9374570,0.08286233,-35.449846,0.08958844,-32.788348
sigma_log_b_hw,0.7746468,0.10475441,7.394884,0.11043156,7.014723
mu_log_b_ch,0.6293038,0.07513541,8.375595,0.08353807,7.533138
sigma_log_b_ch,-0.8555317,0.11434954,-7.481724,0.13405219,-6.382079


In [12]:
# ============================================================
# 7.3 2000-draw
# ============================================================

num_draws <- 2000L

bgw_functions <- make_calcRJ_bgw(num_draws)
calcR_bgw <- bgw_functions$calcR
calcJ_bgw <- bgw_functions$calcJ

jit_time <- system.time({
  invisible(calcR_bgw(start_value_bgw))
  invisible(calcJ_bgw(start_value_bgw))
})[["elapsed"]]

likelihood_eval_time <- system.time({
  initial_prob <- calcR_bgw(start_value_bgw)
})[["elapsed"]]
initial_loglik <- sum(log(initial_prob))

estimation_time <- system.time({
  result_bgw <- estimate_bgw(
    calcR = calcR_bgw,
    calcJ = calcJ_bgw,
    start_value = start_value_bgw
  )
})[["elapsed"]]

stopifnot(isTRUE(result_bgw$hasAnalyticGrad))

postestimation_time <- system.time({
  post_bgw <- postestimation_bgw(
    result = result_bgw,
    calcJ = calcJ_bgw
  )
})[["elapsed"]]

final_loglik <- sum(log(calcR_bgw(result_bgw$estimate)))

timing_bgw <- data.frame(
  draws = num_draws,
  jit_time_sec = jit_time,
  likelihood_eval_time_sec = likelihood_eval_time,
  initial_loglik = initial_loglik,
  final_loglik = final_loglik,
  estimation_time_sec = estimation_time,
  postestimation_time_sec = postestimation_time
)

benchmark_results[[as.character(num_draws)]] <- timing_bgw

print(timing_bgw)
post_bgw$coefficients



BGW using analytic model derivatives supplied by caller...

Valid user-provided bgw_setting names(s) detected...

User-provided value for bgw_settings[["vcHessianMethod]]" is the same as default ("bhhh").

            BetaName InitialBeta(i) D(i)
    1    mu_log_b_tt          -3.00    0
    2 sigma_log_b_tt           0.01    0
    3    mu_log_b_tc          -3.00    0
    4 sigma_log_b_tc           0.01    0
    5    mu_log_b_hw          -3.00    0
    6 sigma_log_b_hw           0.01    0
    7    mu_log_b_ch          -3.00    0
    8 sigma_log_b_ch           0.01    0


    it    nf     F            RELDF    PRELDF    RELDX    MODEL stppar   D*step   NPRELDF
     0     1 2.253780344e+03
     1     2 1.785623676e+03 2.077e-01 9.730e-03 1.85e-03   G   2.40e+01 9.26e-01 5.445e-02
     2     3 1.723282600e+03 3.491e-02 3.445e-02 9.23e-02   G   0.00e+00 2.78e+01 3.445e-02
     3     4 1.523385837e+03 1.160e-01 4.771e-02 1.79e-01   G   0.00e+00 2.73e+01 4.771e-02
     4     6 1.491796445e+0

,Estimate,s.e.,t.rat.(0),Rob.s.e.,Rob.t.rat.(0)
mu_log_b_tt,-1.9888891,0.08810982,-22.572844,0.11423152,-17.411036
sigma_log_b_tt,0.4679013,0.07140161,6.553092,0.06297184,7.430326
mu_log_b_tc,-1.0299883,0.13406833,-7.682562,0.16842844,-6.115287
sigma_log_b_tc,-0.9918310,0.08436447,-11.756502,0.07682152,-12.910849
mu_log_b_hw,-2.9364152,0.08599277,-34.147235,0.09092735,-32.294082
sigma_log_b_hw,0.8471802,0.10851462,7.807061,0.10919207,7.758624
mu_log_b_ch,0.6299276,0.07398257,8.514540,0.08216926,7.666219
sigma_log_b_ch,-0.8360646,0.10473709,-7.982507,0.11554639,-7.235748


In [13]:
# ============================================================
# 7.4 5000-draw
# ============================================================

num_draws <- 5000L

bgw_functions <- make_calcRJ_bgw(num_draws)
calcR_bgw <- bgw_functions$calcR
calcJ_bgw <- bgw_functions$calcJ

jit_time <- system.time({
  invisible(calcR_bgw(start_value_bgw))
  invisible(calcJ_bgw(start_value_bgw))
})[["elapsed"]]

likelihood_eval_time <- system.time({
  initial_prob <- calcR_bgw(start_value_bgw)
})[["elapsed"]]
initial_loglik <- sum(log(initial_prob))

estimation_time <- system.time({
  result_bgw <- estimate_bgw(
    calcR = calcR_bgw,
    calcJ = calcJ_bgw,
    start_value = start_value_bgw
  )
})[["elapsed"]]

stopifnot(isTRUE(result_bgw$hasAnalyticGrad))

postestimation_time <- system.time({
  post_bgw <- postestimation_bgw(
    result = result_bgw,
    calcJ = calcJ_bgw
  )
})[["elapsed"]]

final_loglik <- sum(log(calcR_bgw(result_bgw$estimate)))

timing_bgw <- data.frame(
  draws = num_draws,
  jit_time_sec = jit_time,
  likelihood_eval_time_sec = likelihood_eval_time,
  initial_loglik = initial_loglik,
  final_loglik = final_loglik,
  estimation_time_sec = estimation_time,
  postestimation_time_sec = postestimation_time
)

benchmark_results[[as.character(num_draws)]] <- timing_bgw

print(timing_bgw)
post_bgw$coefficients



BGW using analytic model derivatives supplied by caller...

Valid user-provided bgw_setting names(s) detected...

User-provided value for bgw_settings[["vcHessianMethod]]" is the same as default ("bhhh").

            BetaName InitialBeta(i) D(i)
    1    mu_log_b_tt          -3.00    0
    2 sigma_log_b_tt           0.01    0
    3    mu_log_b_tc          -3.00    0
    4 sigma_log_b_tc           0.01    0
    5    mu_log_b_hw          -3.00    0
    6 sigma_log_b_hw           0.01    0
    7    mu_log_b_ch          -3.00    0
    8 sigma_log_b_ch           0.01    0


    it    nf     F            RELDF    PRELDF    RELDX    MODEL stppar   D*step   NPRELDF
     0     1 2.253779674e+03
     1     2 1.785870019e+03 2.076e-01 9.750e-03 1.84e-03   G   2.40e+01 9.26e-01 5.503e-02
     2     3 1.713709558e+03 4.041e-02 3.528e-02 9.17e-02   G   0.00e+00 2.79e+01 3.528e-02
     3     4 1.630772470e+03 4.840e-02 4.549e-02 1.80e-01   G   0.00e+00 2.65e+01 4.549e-02
     4     5 1.529304104e+0

,Estimate,s.e.,t.rat.(0),Rob.s.e.,Rob.t.rat.(0)
mu_log_b_tt,-1.9960587,0.08891352,-22.449439,0.11434266,-17.456816
sigma_log_b_tt,0.4782523,0.07790630,6.138814,0.07254816,6.592205
mu_log_b_tc,-1.0348794,0.13722439,-7.541512,0.17262557,-5.994937
sigma_log_b_tc,-1.0020089,0.09642903,-10.391155,0.09616051,-10.420170
mu_log_b_hw,-2.9359159,0.08475450,-34.640238,0.09093114,-32.287243
sigma_log_b_hw,0.8127981,0.11493484,7.071817,0.12709787,6.395057
mu_log_b_ch,0.6271487,0.07369831,8.509675,0.08186442,7.660822
sigma_log_b_ch,-0.8308795,0.10277885,-8.084149,0.10975861,-7.570062


In [14]:
# ============================================================
# 7.5 10000-draw
# ============================================================

num_draws <- 10000L

bgw_functions <- make_calcRJ_bgw(num_draws)
calcR_bgw <- bgw_functions$calcR
calcJ_bgw <- bgw_functions$calcJ

jit_time <- system.time({
  invisible(calcR_bgw(start_value_bgw))
  invisible(calcJ_bgw(start_value_bgw))
})[["elapsed"]]

likelihood_eval_time <- system.time({
  initial_prob <- calcR_bgw(start_value_bgw)
})[["elapsed"]]
initial_loglik <- sum(log(initial_prob))

estimation_time <- system.time({
  result_bgw <- estimate_bgw(
    calcR = calcR_bgw,
    calcJ = calcJ_bgw,
    start_value = start_value_bgw
  )
})[["elapsed"]]

stopifnot(isTRUE(result_bgw$hasAnalyticGrad))

postestimation_time <- system.time({
  post_bgw <- postestimation_bgw(
    result = result_bgw,
    calcJ = calcJ_bgw
  )
})[["elapsed"]]

final_loglik <- sum(log(calcR_bgw(result_bgw$estimate)))

timing_bgw <- data.frame(
  draws = num_draws,
  jit_time_sec = jit_time,
  likelihood_eval_time_sec = likelihood_eval_time,
  initial_loglik = initial_loglik,
  final_loglik = final_loglik,
  estimation_time_sec = estimation_time,
  postestimation_time_sec = postestimation_time
)

benchmark_results[[as.character(num_draws)]] <- timing_bgw

print(timing_bgw)
post_bgw$coefficients



BGW using analytic model derivatives supplied by caller...

Valid user-provided bgw_setting names(s) detected...

User-provided value for bgw_settings[["vcHessianMethod]]" is the same as default ("bhhh").

            BetaName InitialBeta(i) D(i)
    1    mu_log_b_tt          -3.00    0
    2 sigma_log_b_tt           0.01    0
    3    mu_log_b_tc          -3.00    0
    4 sigma_log_b_tc           0.01    0
    5    mu_log_b_hw          -3.00    0
    6 sigma_log_b_hw           0.01    0
    7    mu_log_b_ch          -3.00    0
    8 sigma_log_b_ch           0.01    0


    it    nf     F            RELDF    PRELDF    RELDX    MODEL stppar   D*step   NPRELDF
     0     1 2.253780344e+03
     1     2 1.785651884e+03 2.077e-01 9.757e-03 1.84e-03   G   2.41e+01 9.26e-01 5.580e-02
     2     3 1.723039660e+03 3.506e-02 3.428e-02 9.76e-02   G   0.00e+00 2.87e+01 3.428e-02
     3     4 1.529169673e+03 1.125e-01 4.916e-02 1.80e-01   G   0.00e+00 2.66e+01 4.916e-02
     4     6 1.499415035e+0

,Estimate,s.e.,t.rat.(0),Rob.s.e.,Rob.t.rat.(0)
mu_log_b_tt,-1.9951333,0.08879287,-22.469522,0.11473740,-17.388692
sigma_log_b_tt,0.4744678,0.07323001,6.479144,0.06537627,7.257492
mu_log_b_tc,-1.0358486,0.13786064,-7.513737,0.17459179,-5.932974
sigma_log_b_tc,-1.0008887,0.09441541,-10.600905,0.09477248,-10.560964
mu_log_b_hw,-2.9349645,0.08500395,-34.527388,0.09124125,-32.167079
sigma_log_b_hw,0.8157635,0.11716695,6.962403,0.13023686,6.263692
mu_log_b_ch,0.6291557,0.07406289,8.494885,0.08200629,7.672042
sigma_log_b_ch,-0.8418882,0.10250432,-8.213197,0.10765195,-7.820464


In [15]:
# ============================================================
# 8. Summarise and export results
# ============================================================

if (!length(benchmark_results)) {
  stop("Run at least one Section 7 cell before exporting results.")
}

benchmark_timings <- do.call(rbind, benchmark_results)
row.names(benchmark_timings) <- NULL

cpu_name <- tryCatch(
  trimws(system(
    "awk -F: '/^model name/ {print $2; exit}' /proc/cpuinfo",
    intern = TRUE
  )),
  error = function(e) NA_character_
)

benchmark_summary <- data.frame(
  package = "ANVL",
  implementation = "anvl_bgw_cpu",
  backend = "CPU",
  device = cpu_name,
  optimiser = "BGW",
  draws = benchmark_timings$draws,
  seed = seed,
  jit_time_sec = benchmark_timings$jit_time_sec,
  likelihood_eval_time_sec = benchmark_timings$likelihood_eval_time_sec,
  initial_loglik = benchmark_timings$initial_loglik,
  final_loglik = benchmark_timings$final_loglik,
  mean_estimation_time_sec = benchmark_timings$estimation_time_sec,
  postestimation_time_sec = benchmark_timings$postestimation_time_sec,
  stringsAsFactors = FALSE
)

benchmark_summary <- benchmark_summary[order(benchmark_summary$draws), ]
row.names(benchmark_summary) <- NULL

time_columns <- c(
  "jit_time_sec", "likelihood_eval_time_sec",
  "mean_estimation_time_sec", "postestimation_time_sec"
)
benchmark_summary[time_columns] <- lapply(
  benchmark_summary[time_columns],
  round,
  digits = 4
)
benchmark_summary[c("initial_loglik", "final_loglik")] <- lapply(
  benchmark_summary[c("initial_loglik", "final_loglik")],
  round,
  digits = 2
)

if (!requireNamespace("writexl", quietly = TRUE)) {
  install.packages("writexl")
}
writexl::write_xlsx(
  list(benchmark_summary = benchmark_summary),
  path = summary_file
)

print(benchmark_summary)
cat("Excel file saved to:", summary_file, "\n")


Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)



  package implementation backend                         device optimiser draws
1    ANVL   anvl_bgw_cpu     CPU Intel(R) Xeon(R) CPU @ 2.20GHz       BGW   500
2    ANVL   anvl_bgw_cpu     CPU Intel(R) Xeon(R) CPU @ 2.20GHz       BGW  1000
3    ANVL   anvl_bgw_cpu     CPU Intel(R) Xeon(R) CPU @ 2.20GHz       BGW  2000
4    ANVL   anvl_bgw_cpu     CPU Intel(R) Xeon(R) CPU @ 2.20GHz       BGW  5000
5    ANVL   anvl_bgw_cpu     CPU Intel(R) Xeon(R) CPU @ 2.20GHz       BGW 10000
  seed jit_time_sec likelihood_eval_time_sec initial_loglik final_loglik
1  123        1.687                    0.031       -2253.78     -1445.69
2  123        1.211                    0.050       -2253.78     -1445.08
3  123        1.349                    0.089       -2253.78     -1444.09
4  123        1.931                    0.252       -2253.78     -1444.67
5  123        2.905                    0.453       -2253.78     -1444.56
  mean_estimation_time_sec postestimation_time_sec
1                    2.532     